# Conv LSTM Minimal Reproducible Example

This notebook provides a minimal reproducible example of using Optuna with an Conv-LSTM sequence to sequence model in pytorch. We use the M5 dataset as a test case, training one local model per time series. There are 30,490 unique series in the dataset. There are between 124 and 1969 observations per series. Over 25,000 series have more than 1000 observations.

## Dataset
We will use Nixtla's datasetforecast library to load the M5 dataset. This saves locally for now but we will need to reproduce the exact customer data load for benchmarking.

In [1]:
# Get the m5 dataset
# TODO: reproduce exact customer data load for benchmarking
from datasetsforecast.m5 import M5
m5_dataset = M5(source_url='https://github.com/Nixtla/m5-forecasts/raw/main/datasets/m5.zip')
sales, calendar, heirarchy = m5_dataset.load(directory='../data')

There is plenty of data here. Almost all of the 30k series have enough data to train with, and we are likely talking billions of rows when we window (see below).

In [2]:
import plotly.express as px
import pandas as pd
series_lengths = pd.DataFrame(sales.groupby('unique_id').size(), columns=['length'])
series_lengths = series_lengths.reset_index()
fig = px.histogram(series_lengths, x='length', nbins=30, width=500, height=500, title='Series Lengths')
fig.show()

/var/folders/2x/pf13chqx4614qjlmjdgmndv00000gp/T/ipykernel_46787/1641167039.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  series_lengths = pd.DataFrame(sales.groupby('unique_id').size(), columns=['length'])


In [267]:
# select a five IDs for debugging
# TODO: decide on benchmarking scale
uids = sales['unique_id'].sample(5, random_state=4)
y_df = sales.query('unique_id in @uids')

from statsforecast import StatsForecast
StatsForecast.plot(y_df,engine='plotly')

We can leverage utilsforecast to generate a rolling window of data for neural network training off a single series. This is perfect for doing sequence to sequence modelling because it generates an input with a specified size (e.g. 104 days) and a target with a specified size (e.g. 52 days), along with the cutoff date. The cutoff data is the maximum data in the input data (the last date observed) We can use this to generate a large delta table or a pytorch dataset.

In [4]:
from dbtsr.utils import auto_backtest_splits
combined = auto_backtest_splits(y_df, h=52, freq='D', id_col='unique_id', time_col='ds', step_size=1, input_size=104)
# focus on the first group for debugging
first_grp = combined[combined['unique_id'] == combined['unique_id'].iloc[0]]

We can then take our Delta friendly format and convert it into a sequence to sequence dataset. This dataset follows the standard pytorch tensor format of (batch, channel, sequence length).

In [10]:
# Easily optimized - naive and slow
from dbtsr.utils import create_seq2seq_dataset
input, target = create_seq2seq_dataset(first_grp, input_size=104, output_size=52)
input.shape, target.shape

/Users/scott.mckean/Repos/timeseries_research/src/dbtsr/utils.py:65: UserWarning:

Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:281.)



(torch.Size([1813, 1, 104]), torch.Size([1813, 1, 52]))

Now that we have an input and target, we can generate an encoder / decoder framework. We are going to use a sequence to seuqence model with 1D convolutions and LSTM cells.

In [140]:
# Split data into train/validation sets (90/10 split)
train_size = int(0.9 * len(input))
train_input = input[:train_size]
train_target = target[:train_size]
val_input = input[train_size:]
val_target = target[train_size:]

This is the bulk of the pytorch code. We define an encoder and decoder, and then a sequence to sequence model. We then define a training loop and a validation loop. A couple gotchas while developing - the LSTM hidden sizes need to be the same on the encoder and decoder. 

*TODO: add exogenous features*

In [261]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(
            self, 
            input_size=1, 
            lstm_hidden_size=64, 
            lstm_num_layers=2, 
            lstm_dropout=0.1,
            conv_kernel_size=3,
            num_conv_layers=2,
            activation=nn.Identity()
            ):
        super().__init__()
        self.lstm_hidden_size = lstm_hidden_size
        self.lstm_num_layers = lstm_num_layers
        self.lstm_dropout = lstm_dropout
        self.conv_kernel_size = conv_kernel_size
        self.activation = activation
        
        # CONVOLUTIONAL ENCODER
        # CONV input: [batch, features, input_seq_length]
        self.conv_layers = nn.ModuleList()
        
        # First conv layer takes raw input
        self.conv_layers.append(nn.Conv1d(
            in_channels=input_size,
            out_channels=lstm_hidden_size,
            kernel_size=conv_kernel_size,
            padding="same"
        ))
        
        # Additional conv layers if specified
        for _ in range(num_conv_layers - 1):
            self.conv_layers.append(nn.Conv1d(
                in_channels=lstm_hidden_size,
                out_channels=lstm_hidden_size, 
                kernel_size=conv_kernel_size,
                padding="same"
            ))
            self.conv_layers.append(self.activation)

        # LSTM ENCODER
        # LSTM input: [batch, input_seq_length, features]
        self.lstm = nn.LSTM(
            input_size=lstm_hidden_size,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            dropout=lstm_dropout,
            batch_first=True
        )
    
    def forward(self, x):
        # x shape: [batch_size, features, input_seq_length]
        for conv_layer in self.conv_layers:
            x = conv_layer(x)

        # LSTM input shape: [batch_size, input_seq_length, features]
        x = x.permute(0, 2, 1)
        
        # lstm_out shape: [batch_size, input_seq_length, hidden_size]
        # hidden/cell shape: [num_layers, batch_size, hidden_size]
        lstm_out, (hidden, cell) = self.lstm(x)
        
        return lstm_out, hidden, cell

class Decoder(nn.Module):
    def __init__(
            self, 
            input_size=1, 
            lstm_hidden_size=64, 
            lstm_num_layers=2, 
            lstm_dropout=0.1,
            activation=nn.Identity(),
            linear_hidden_sizes=[],
            ):
        super().__init__()
        self.lstm_hidden_size = lstm_hidden_size
        self.lstm_num_layers = lstm_num_layers
        self.lstm_dropout = lstm_dropout
        self.activation = activation
        
        # LSTM DECODER
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            dropout=lstm_dropout,
            batch_first=True
        )
        
       # LINEAR DECODER
       # Ensure at least one linear layer with final output size of 1
        if linear_hidden_sizes:
            self.linear_layers = nn.ModuleList()
            self.linear_layers.append(nn.Linear(lstm_hidden_size, linear_hidden_sizes[0]))
            self.linear_layers.append(self.activation)
            for i in range(len(linear_hidden_sizes)-1):
                self.linear_layers.append(nn.Linear(linear_hidden_sizes[i], linear_hidden_sizes[i+1]))
                self.linear_layers.append(self.activation)
            self.linear_layers.append(nn.Linear(linear_hidden_sizes[-1], 1))
        else:
            self.linear_layers = nn.ModuleList()
            self.linear_layers.append(nn.Linear(lstm_hidden_size, 1))
    
    def forward(self, x, hidden, cell):
        # x shape: [batch_size, 1, 1] (single time step)
        # hidden/cell shape: [num_layers, batch_size, hidden_size]
        # lstm_out shape: [batch_size, 1, hidden_size]
        lstm_out, (hidden, cell) = self.lstm(x, (hidden, cell))
        x = self.activation(lstm_out)
        
        # output shape: [batch_size, 1, 1]
        for layer in self.linear_layers:
            x = layer(x)

        return x, hidden, cell

class Seq2SeqModel(nn.Module):
    def __init__(
            self, 
            lstm_hidden_size=64, 
            lstm_num_layers=2, 
            enc_lstm_dropout=0.1, 
            enc_activation="silu",
            enc_conv_kernel_size=3,
            enc_conv_layers=2,
            dec_lstm_dropout=0.1, 
            dec_activation="silu",
            dec_linear_hidden_sizes=[64, 32],
            teacher_forcing_ratio=0.5,
            ):
        super().__init__()
        self.teacher_forcing_ratio = teacher_forcing_ratio
        
        self.activation_map = {
            'sigmoid': nn.Sigmoid(),
            'relu': nn.ReLU(),
            'silu': nn.SiLU(),
            'leakyrelu': nn.LeakyReLU(),
            'identity': nn.Identity()
        }

        self.encoder = Encoder(
            input_size=1, 
            lstm_hidden_size=lstm_hidden_size, 
            lstm_num_layers=lstm_num_layers, 
            lstm_dropout=enc_lstm_dropout,
            conv_kernel_size=enc_conv_kernel_size,
            num_conv_layers=enc_conv_layers,
            activation=self.activation_map.get(enc_activation, nn.Identity())
            )
        
        self.decoder = Decoder(
            input_size=1, 
            lstm_hidden_size=lstm_hidden_size, 
            lstm_num_layers=lstm_num_layers, 
            lstm_dropout=dec_lstm_dropout,
            activation=self.activation_map.get(dec_activation, nn.Identity()),
            linear_hidden_sizes=dec_linear_hidden_sizes
            )

    def forward(self, x, y, out_seq_length, teacher_forcing_ratio=None):
        batch_size = x.shape[0]
        if teacher_forcing_ratio is None:
            teacher_forcing_ratio = self.teacher_forcing_ratio
        
        # Encode input sequence
        _, hidden, cell = self.encoder(x)
        
        # Initialize decoder input with last value of input sequence
        decoder_input = x[:, :, -1:].permute(0, 2, 1)  # shape: [batch_size, 1, 1]
        
        # Initialize outputs tensor
        outputs = torch.zeros(batch_size, 1, out_seq_length).to(x.device)
        
        # Generate sequence
        for t in range(out_seq_length):
            # Generate prediction
            output, hidden, cell = self.decoder(decoder_input, hidden, cell)
            outputs[:, :, t] = output.squeeze(1)
            
            # Teacher forcing
            if torch.rand(1) < teacher_forcing_ratio and t < out_seq_length - 1:
                decoder_input = y[:, :, t:t+1].permute(0, 2, 1)
            else:
                decoder_input = output
        
        return outputs

Here is a single test run of the model. We can see that the model is able to learn the sequence to sequence problem, but performs pretty poorly (overregularized and doesn't capture any seasonality). We use days in the M5 problem instead of weeks, but the scale should be sufficient.

In [262]:
# Hyperparameters
learning_rate = 0.005
batch_size = 500
num_epochs = 5
target_seq_length = train_target.shape[2]

# Initialize model
model = Seq2SeqModel(
    lstm_hidden_size=128,
    lstm_num_layers=2,
    enc_lstm_dropout=0.1,
    enc_activation="silu",
    enc_conv_layers=5,
    dec_lstm_dropout=0.1,
    dec_activation="silu",
    dec_linear_hidden_sizes=[64, 32],
    teacher_forcing_ratio=0.5
)

# Test forward pass
outputs = model(train_input, train_target, out_seq_length=target_seq_length)
print(f"Input shape: {train_input.shape}")
print(f"Output shape: {outputs.shape}")

# Training setup
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop with validation
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    # Process in batches
    for i in range(0, len(train_input), batch_size):
        batch_input = train_input[i:i + batch_size]
        batch_target = train_target[i:i + batch_size]
        
        optimizer.zero_grad()
        output = model(batch_input, batch_target, out_seq_length=target_seq_length)
        loss = criterion(output, batch_target)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    # Validation step
    model.eval()
    with torch.no_grad():
        val_output = model(val_input, val_target, out_seq_length=target_seq_length, teacher_forcing_ratio=0.0)
        val_loss = criterion(val_output, val_target)
        
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'Training Loss: {total_loss/len(train_input):.4f}')
    print(f'Validation Loss: {val_loss.item():.4f}')

Input shape: torch.Size([1631, 1, 104])
Output shape: torch.Size([1631, 1, 52])
Epoch [1/5]
Training Loss: 0.0919
Validation Loss: 26.6459
Epoch [2/5]
Training Loss: 0.0906
Validation Loss: 26.0739
Epoch [3/5]
Training Loss: 0.0871
Validation Loss: 24.0341
Epoch [4/5]
Training Loss: 0.0801
Validation Loss: 20.8013
Epoch [5/5]
Training Loss: 0.0713
Validation Loss: 17.4565


Let's take a look at the amazing predictions.

In [268]:
# Prediction
model.eval()
with torch.no_grad():
    pred = model(val_input, val_target, out_seq_length=target_seq_length, teacher_forcing_ratio=0.0)
    pred_loss = criterion(val_output, val_target)

In [270]:
import plotly.express as px
import numpy as np

first_pred = pred[-1, 0, :].numpy()
first_target = val_target[-1, 0, :].numpy()
time_steps = list(range(len(first_pred)))

df = pd.DataFrame({
    'Time Step': time_steps * 2,  # Duplicate time steps for both series
    'Value': np.concatenate([first_pred, first_target]),
    'Type': ['Prediction'] * len(first_pred) + ['Target'] * len(first_target)
})

px.line(df, x='Time Step', y='Value', color='Type', title='Prediction vs Target for First Series')

Okay, now to get down to some of the benchmarking - I have a naive Optuna run with a TPESampler. Would like some comments and validation on this as we start to move into Databricks for optimization.

In [266]:
import optuna
from optuna.pruners import HyperbandPruner
from optuna.samplers import TPESampler

# Fixed parameters
target_seq_length = train_target.shape[2]

def objective(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    batch_size = trial.suggest_int('batch_size', 30, 250)
    # kept this constant for fair comparisons
    num_epochs = 3
    
    lstm_hidden_size = trial.suggest_int('enc_lstm_hidden_size', 32, 256)
    teacher_forcing_ratio = trial.suggest_float('teacher_forcing_ratio', 0.0, 0.8)
    lstm_num_layers = trial.suggest_int('enc_lstm_num_layers', 2, 5)
    enc_lstm_dropout = trial.suggest_float('enc_lstm_dropout', 0.0, 0.5)
    enc_conv_layers = trial.suggest_int('enc_conv_layers', 1, 5)
    dec_lstm_dropout = trial.suggest_float('dec_lstm_dropout', 0.0, 0.5)
    
    # TODO: improve this sampling
    dec_linear_hidden_sizes = [
        trial.suggest_int('dec_linear_hidden1', 16, 256),
        trial.suggest_int('dec_linear_hidden2', 16, 128)
    ]
    
    # 
    model = Seq2SeqModel(
        lstm_hidden_size=lstm_hidden_size,
        lstm_num_layers=lstm_num_layers,
        enc_lstm_dropout=enc_lstm_dropout,
        enc_activation="silu",
        enc_conv_layers=enc_conv_layers,
        dec_lstm_dropout=dec_lstm_dropout,
        dec_activation="silu",
        dec_linear_hidden_sizes=dec_linear_hidden_sizes,
        teacher_forcing_ratio=teacher_forcing_ratio
    )
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for i in range(0, len(train_input), batch_size):
            batch_input = train_input[i:i + batch_size]
            batch_target = train_target[i:i + batch_size]
            
            optimizer.zero_grad()
            output = model(batch_input, batch_target, out_seq_length=target_seq_length)
            loss = criterion(output, batch_target)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        model.eval()
        with torch.no_grad():
            val_output = model(val_input, val_target, out_seq_length=target_seq_length, teacher_forcing_ratio=0.0)
            val_loss = criterion(val_output, val_target)
        
        # pruning step
        trial.report(val_loss.item(), epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
            
    return val_loss.item()

# Optuna Hyperparameter Optimization
# TODO: add GPU support and test on Databricks runtimes
study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(),
    pruner=HyperbandPruner()
)

study.optimize(objective, n_trials=10)

#TODO: replace with mlflow
print("Best trial:")
trial = study.best_trial
print(f"  Value: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2025-01-02 23:06:35,670] A new study created in memory with name: no-name-80b67e51-8824-455f-8c55-d3d222813d65
[I 2025-01-02 23:07:21,484] Trial 0 finished with value: 11.834904670715332 and parameters: {'learning_rate': 0.002512220186712167, 'batch_size': 71, 'enc_lstm_hidden_size': 205, 'teacher_forcing_ratio': 0.653757738490787, 'enc_lstm_num_layers': 5, 'enc_lstm_dropout': 0.3111656761526665, 'enc_conv_layers': 4, 'dec_lstm_dropout': 0.3251680267975906, 'dec_linear_hidden1': 232, 'dec_linear_hidden2': 36}. Best is trial 0 with value: 11.834904670715332.
[I 2025-01-02 23:07:29,359] Trial 1 finished with value: 19.843271255493164 and parameters: {'learning_rate': 0.03698494304585105, 'batch_size': 228, 'enc_lstm_hidden_size': 118, 'teacher_forcing_ratio': 0.7935900461685392, 'enc_lstm_num_layers': 3, 'enc_lstm_dropout': 0.2898158434788566, 'enc_conv_layers': 1, 'dec_lstm_dropout': 0.07744713998610947, 'dec_linear_hidden1': 256, 'dec_linear_hidden2': 42}. Best is trial 0 with value

Best trial:
  Value: 10.458379745483398
  Params: 
    learning_rate: 0.002244079706913082
    batch_size: 72
    enc_lstm_hidden_size: 187
    teacher_forcing_ratio: 0.19245007706766126
    enc_lstm_num_layers: 5
    enc_lstm_dropout: 0.4861336042971853
    enc_conv_layers: 5
    dec_lstm_dropout: 0.4150365149307773
    dec_linear_hidden1: 173
    dec_linear_hidden2: 25
